# Stale semantic prompt — nhánh T, hai GPU
**T4×2 · Internet ON · Save & Run All.** Mỗi notebook chỉ chạy một nhánh.
C và T cùng seed/order,TRAIN10k,2.000aux,1.250updates,batch tổng8,LR5e-6.
C:main CE +.1KL neo A +.1neutral CE +.1visual contrast; T thêm .1stale loss.
Dùng nguyên cache đã chuẩn bị; không mã hóa lại ảnh hay prompt, không chọn ca active.
Raw greedy A và end trên dev1000; checkpoint cuối duy nhất. Không SCST/sweep/test.
**Argon chưa phản biện độc lập**; user đã đồng ý thử C/T có giới hạn sau quyết định Astra.
Gắn output `ducanh2403/stale`, checkpoint A, prompt-cache .pt, visual-cache, MSCOCO JSON.
Không cần ảnh gốc, prompt JSON hoặc UPT cache. Tự tìm receipt đúng trong thư mục con.
Stage main inputs có thể tốn I/O đáng kể, cần khoảng8GiB trống; không nằm trong profile6phút.

In [ ]:
from pathlib import Path
ARM='T'
CACHE_ROOT=Path('/kaggle/input/notebooks/ducanh2403/stale')
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/stale_prompt_CT_T')
STAGE=Path('/tmp/stale_prompt_CT_T_main.h5')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON):assert p.is_file(),p
for p in (CACHE_ROOT,VISUAL_CACHE):assert p.is_dir(),p
assert not OUTPUT.exists() and not STAGE.exists(),'Preserve previous output; use fresh paths/session.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-stale-ct-t-'))/'repo'
LOG = Path('/tmp/upt_hidden_stale-ct-t.log')
PROJECT_COMMIT = '335b136d335fa92ae9be3e4b86772044b770e7f2'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'run_stale.py': 'c8572f5406d544de85e6a1920e1e790f690f11bc741160cf3f0815a73c430154', 'pilot_contract.py': '21b670f79e6614eed9bb26ac502e6fb4ba08163546bf9ad91b5d604128136094', 'test_contract.py': '3709487515c6cce01cccc38880b85867c4d582110b8f04b5d6cb6d884fcb598d', 'test_pilot_ddp.py': 'c852e7a525eac20465b95aa0f58c5afdbc334b8fdffb3de2537bd42b28e538de', 'profile_stale.py': '08ed37d60dda8f89a8671fcaed7dd749a10b33baf5ea1a126ee0457bf33dc1c8', 'stale_data.py': '21f4962a51380b77d7c98bf6f7ecc89386b30eecb46a4d5f80fcc49e0a8fdcfe', 'test_stale_data.py': '54dede035da17d24bdcba5a063ffabd88efff2b304dd9e18a704753925fb2c5f', 'test_stale_runtime.py': 'a43b98378411af1853121d6207817f8b5cb97d0f4e612914a66653b202f361c3', 'run_prepare.py': '46b3e9cb6f4e5d1e4790b13dbdb976117898dc8610488fb1b6c7fa0d7291d3f8', 'run_diagnostic.py': '382653c64ecc7047f908ae717d5d963efa177767ad8db4774093207e286ff731', 'diagnostic_data.py': '15c0eab863ad6500436d25372e34e3395266774838db39b2bdb8e402ff150312', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2','h5py','numpy','Pillow','tqdm','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=Path('/tmp/stale-ct-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
import torch
assert torch.cuda.device_count()==2,'Select T4x2 for each one-arm notebook.'
run([sys.executable,'-u','test_contract.py'],cwd=REPO)
run([sys.executable,'-u','test_stale_data.py'],cwd=REPO)
run([sys.executable,'-u','test_stale_runtime.py'],cwd=REPO)
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0','test_pilot_ddp.py'],cwd=REPO)


In [ ]:
cfg=dict(arm=ARM,cache_root=str(CACHE_ROOT),checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),chair=str(CHAIR),output=str(OUTPUT),stage=str(STAGE),implementation_sha256=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
CONFIG=Path('/tmp/stale_ct_'+ARM+'.json');CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_stale.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0','run_stale.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_stale.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        run([sys.executable,'-u','run_stale.py','archive','--config',PROTOCOL if PROTOCOL.is_file() else CONFIG],cwd=REPO)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)


Gửi ZIP review của cả C và T; giữ riêng checkpoint.bin và cache gốc.
ZIP có log/protocol/predictions/per-imageCIDEr/CHAIR, không có HDF5 hoặc checkpoint lớn.
Một nhánh hoàn tất chưa đủ tuyên bố thắng: cần so T−A và T−C bằng pairedCI,
CHAIR/recall và kiểm prompt abandonment/artifact trước claim cơ chế.
Dừng nếu hash/binding/frozen state lỗi,nonfinite/OOM hoặc projection thực vượt2h/arm.
Không dùng interim dev để sửa loss/chọn checkpoint/đổi seed.